# CoinMarketCap API - Real-Time Cryptocurrency Data Pipeline

This notebook is configured to fetch real-time cryptocurrency data from CoinMarketCap API.

## ✅ Configuration Status
* **API Endpoint**: `https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest`
* **Authentication**: X-CMC_PRO_API_KEY header
* **API Key**: Configured ✓
* **Test Status**: Successfully fetched 100 cryptocurrency records

## 📊 Data Retrieved
The API returns comprehensive cryptocurrency data including:
* Basic info: ID, name, symbol, slug
* Supply metrics: circulating supply, total supply, max supply
* Market data: CMC rank, market pairs, market cap
* Price data: USD prices, 24h volume, price changes
* Timestamps: date_added, last_updated, ingested_at

## 🚀 Next Steps

### Option 1: Save to Delta Table (Recommended)
Run **Cell 3** to save the cryptocurrency data to a Delta table for analysis and dashboarding.

### Option 2: Set Up Scheduled Ingestion
Use **Cell 2** to configure continuous real-time data ingestion:
* **Polling Loop**: For testing (every 5-15 minutes)
* **Incremental Upsert**: For production (updates existing records)

### Option 3: Secure Your API Key
Run **Cell 4** to move your API key to Databricks Secrets (best practice for production).

## 📝 Usage Recommendations
* **API Rate Limits**: Professional plan = 333 calls/day (~1 call every 4 minutes)
* **Update Frequency**: CoinMarketCap updates every 1-2 minutes
* **Optimal Schedule**: Run every 5-10 minutes for near real-time data
* **Table Strategy**: Use upsert pattern to maintain latest prices without duplicates

In [0]:
import requests
import pandas as pd
from pyspark.sql import DataFrame
import json
from datetime import datetime

# CoinMarketCap API Configuration
API_ENDPOINT = "https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest"
API_TOKEN = "15082d12facc473387f6c0c53728ea1a"

# CoinMarketCap uses X-CMC_PRO_API_KEY authentication
headers = {
    "Accepts": "application/json",
    "X-CMC_PRO_API_KEY": API_TOKEN
}

# Optional: Query parameters for filtering data
params = {
    "limit": 100,
    "start": 1,
    "convert": "USD"
}

# Make API request
try:
    response = requests.get(
        API_ENDPOINT,
        headers=headers,
        params=params,
        timeout=30
    )
    
    # Check if request was successful
    response.raise_for_status()
    
    # Parse JSON response
    data = response.json()
    
    # CoinMarketCap returns data in {"data": [...]} format
    # Use pandas for better handling of nested JSON and mixed types
    if isinstance(data, dict) and "data" in data:
        pandas_df = pd.json_normalize(data["data"])
    elif isinstance(data, list):
        pandas_df = pd.json_normalize(data)
    else:
        pandas_df = pd.json_normalize(data)
    
    # Convert pandas DataFrame to Spark DataFrame
    df = spark.createDataFrame(pandas_df)
    
    # Add ingestion timestamp
    from pyspark.sql.functions import current_timestamp
    df = df.withColumn("ingested_at", current_timestamp())
    
    # Display results
    display(df)
    
    print(f"✓ Successfully fetched {df.count()} cryptocurrency records")
    print(f"Columns: {', '.join(df.columns[:10])}...")
    
except requests.exceptions.HTTPError as e:
    print(f"HTTP Error: {e}")
    print(f"Response: {response.text}")
except requests.exceptions.RequestException as e:
    print(f"Request Error: {e}")
except Exception as e:
    print(f"Error: {e}")

In [0]:
# Save cryptocurrency data to Delta table under main catalog
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("main.default.crypto_market_data")

print(f"✓ Successfully saved {df.count()} records to main.default.crypto_market_data")
print(f"\nTo query the table, use:")
print("spark.sql('SELECT * FROM main.default.crypto_market_data LIMIT 10').display()")

In [0]:
# OPTION 1: Simple Polling Loop for Crypto Data (for testing/development)
# Use this for quick testing, but prefer scheduled jobs for production

import time
from datetime import datetime

def ingest_crypto_realtime(api_endpoint, headers, interval_seconds=300):
    """
    Continuously fetch cryptocurrency data from CoinMarketCap at regular intervals
    
    Args:
        api_endpoint: CoinMarketCap API URL
        headers: Authentication headers
        interval_seconds: Time between requests (default: 300 = 5 minutes)
    """
    while True:
        try:
            print(f"[{datetime.now()}] Fetching cryptocurrency data...")
            
            params = {"limit": 100, "start": 1, "convert": "USD"}
            response = requests.get(api_endpoint, headers=headers, params=params, timeout=30)
            response.raise_for_status()
            
            data = response.json()
            
            # Convert to DataFrame
            pandas_df = pd.json_normalize(data["data"])
            df = spark.createDataFrame(pandas_df)
            
            from pyspark.sql.functions import current_timestamp
            df = df.withColumn("ingested_at", current_timestamp())
            
            # Append to Delta table
            df.write.format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable("main.default.crypto_realtime_data")
            
            print(f"✓ Ingested {df.count()} cryptocurrency records")
            
        except Exception as e:
            print(f"✗ Error: {e}")
        
        # Wait before next fetch
        time.sleep(interval_seconds)

# # Usage (uncomment to run):
# ingest_crypto_realtime(API_ENDPOINT, headers, interval_seconds=300)


# OPTION 2: Incremental Load (Recommended for Real-Time Crypto)
# Upsert pattern - updates existing coins and adds new ones

def get_last_update_time(table_name):
    """Get the most recent data update timestamp"""
    try:
        result = spark.sql(f"""
            SELECT MAX(ingested_at) as last_time 
            FROM {table_name}
        """).collect()[0]["last_time"]
        return result
    except:
        return None

def ingest_crypto_incremental(api_endpoint, headers, table_name="main.default.crypto_market_data"):
    """
    Fetch cryptocurrency data and upsert to Delta table
    Updates existing records based on coin ID
    """
    from delta.tables import DeltaTable
    
    last_time = get_last_update_time(table_name)
    
    if last_time:
        print(f"Last update: {last_time}")
    else:
        print("First run - fetching initial data")
    
    # Fetch latest data
    params = {"limit": 100, "start": 1, "convert": "USD"}
    response = requests.get(api_endpoint, headers=headers, params=params)
    response.raise_for_status()
    
    data = response.json()
    pandas_df = pd.json_normalize(data["data"])
    new_df = spark.createDataFrame(pandas_df)
    
    from pyspark.sql.functions import current_timestamp
    new_df = new_df.withColumn("ingested_at", current_timestamp())
    
    # Upsert to Delta table
    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        
        delta_table.alias("target").merge(
            new_df.alias("source"),
            "target.id = source.id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        
        print(f"✓ Updated {new_df.count()} cryptocurrency records")
    else:
        new_df.write.format("delta").saveAsTable(table_name)
        print(f"✓ Created new table with {new_df.count()} records")

# # Usage:
# ingest_crypto_incremental(API_ENDPOINT, headers)


print("\n📋 For Production Real-Time Crypto Data Ingestion:")
print("1. Use Databricks Jobs to schedule this notebook (every 5-15 minutes)")
print("2. CoinMarketCap updates prices every 1-2 minutes, so 5-10 min intervals are optimal")
print("3. Use the upsert pattern (OPTION 2) to maintain latest prices without duplicates")
print("4. Consider API rate limits: Professional plan allows 333 calls/day")

In [0]:
# PATTERN 1: Save Cryptocurrency Data to Delta Table
# Uncomment and run after fetching data

# # Create or append to Delta table
# df.write.format("delta") \
#     .mode("append") \
#     .option("mergeSchema", "true") \
#     .saveAsTable("main.default.crypto_market_data")

# PATTERN 2: Upsert Pattern (Merge for Updates) - Updates existing crypto records
from delta.tables import DeltaTable

def upsert_crypto_data(new_df, target_table="main.default.crypto_market_data"):
    """
    Upsert cryptocurrency data based on id (coin ID)
    Updates existing records and inserts new ones
    """
    # Check if table exists
    if spark.catalog.tableExists(target_table):
        delta_table = DeltaTable.forName(spark, target_table)
        
        # Merge based on cryptocurrency ID
        delta_table.alias("target").merge(
            new_df.alias("source"),
            "target.id = source.id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        
        print(f"✓ Upserted {new_df.count()} cryptocurrency records")
    else:
        new_df.write.format("delta").saveAsTable(target_table)
        print(f"✓ Created new table with {new_df.count()} records")

# # Usage:
# upsert_crypto_data(df)


# PATTERN 3: Pagination for Large Datasets (CoinMarketCap)
def fetch_all_crypto_pages(base_url, headers, page_size=100, max_coins=5000):
    """
    Fetch multiple pages from CoinMarketCap API
    
    Args:
        base_url: API endpoint
        headers: Authentication headers
        page_size: Records per page (max 5000 for CoinMarketCap)
        max_coins: Maximum number of coins to fetch
    """
    all_data = []
    start = 1
    
    while start <= max_coins:
        params = {
            "start": start,
            "limit": page_size,
            "convert": "USD"
        }
        
        response = requests.get(base_url, headers=headers, params=params)
        response.raise_for_status()
        
        data = response.json()
        records = data.get("data", [])
        
        if not records:
            break
        
        all_data.extend(records)
        print(f"Fetched {len(records)} records (total: {len(all_data)})")
        
        # Break if we got fewer records than requested (last page)
        if len(records) < page_size:
            break
        
        start += page_size
    
    return all_data

# # Usage:
# all_crypto_data = fetch_all_crypto_pages(API_ENDPOINT, headers, page_size=100, max_coins=500)
# pandas_df_complete = pd.json_normalize(all_crypto_data)
# df_complete = spark.createDataFrame(pandas_df_complete)

print("Real-time crypto data ingestion patterns ready to use!")

In [0]:
# BEST PRACTICE: Store CoinMarketCap API key securely using Databricks Secrets
# This prevents hardcoding sensitive credentials in notebooks

# Step 1: Create a secret scope (run this once in Databricks CLI or UI)
# databricks secrets create-scope --scope coinmarketcap_secrets

# Step 2: Store your API key (run once in Databricks CLI)
# databricks secrets put --scope coinmarketcap_secrets --key api_key
# Then paste your key: 15082d12facc473387f6c0c53728ea1a

# Step 3: Retrieve API key securely in your notebook (uncomment after creating the secret)
# API_TOKEN = dbutils.secrets.get(scope="coinmarketcap_secrets", key="api_key")

# Step 4: Use the token in your API calls
# headers = {
#     "Accepts": "application/json",
#     "X-CMC_PRO_API_KEY": API_TOKEN
# }

print("\nTo set up Databricks Secrets for your CoinMarketCap API key:")
print("\nOption 1 - Via Databricks CLI:")
print("1. Install CLI: pip install databricks-cli")
print("2. Configure: databricks configure --token")
print("3. Create scope: databricks secrets create-scope --scope coinmarketcap_secrets")
print("4. Add secret: databricks secrets put --scope coinmarketcap_secrets --key api_key")
print("\nOption 2 - Via Databricks UI:")
print("1. Go to Settings > Developer > Manage Secrets")
print("2. Create a new scope named 'coinmarketcap_secrets'")
print("3. Add a secret with key 'api_key' and your CoinMarketCap API key as the value")
print("\nAfter creating the secret, uncomment lines 14-19 in this cell to use it.")